# Day 16: Disaggregation — Prefill/Decode Split
> *Inference Engineering* — Chapter 5.5 | Philip Kiely, Baseten Books 2026

**Layer:** Runtime | **Prerequisite:** Day 10 (Dynamo), Day 15 (Model Parallelism)


## Concept Overview

Chapter 5.5 extends the disaggregation concept from Day 10 (Dynamo architecture) to the algorithmic and systems design level. Prefill-decode disaggregation is not just about separate workers — it's about optimal resource allocation given their fundamentally different compute profiles. Prefill workers need high compute (H100s), decode workers need high memory bandwidth. The optimal ratio of prefill:decode workers depends on input:output length ratio, SLO requirements, and model size. This day focuses on modeling and optimizing that ratio.


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Prefill and Decode Time Models

Building analytical models for prefill and decode time as functions of model config and hardware.


In [ ]:
class InferenceTimeModel:
    def __init__(self, model_name, num_layers, d_model, num_heads, d_head, num_kv_heads,
                 peak_flops_tflops, hbm_bw_gb_s, dtype_bytes=2):
        self.name = model_name
        self.L = num_layers
        self.D = d_model
        self.H = num_heads
        self.Hkv = num_kv_heads
        self.dh = d_head
        self.peak_flops = peak_flops_tflops * 1e12
        self.hbm_bw = hbm_bw_gb_s * 1e9
        self.dtype = dtype_bytes

    def prefill_time_ms(self, seq_len, batch=1):
        """Compute-bound prefill: dominated by FLOPs."""
        # Attention FLOPs
        attn = self.L * (4 * batch * seq_len * self.D**2 +
                         2 * batch * seq_len**2 * self.D)
        # FFN FLOPs
        ffn = self.L * 2 * batch * seq_len * self.D * 4 * self.D
        total_flops = attn + ffn
        return total_flops / self.peak_flops * 1000

    def decode_time_ms(self, kv_len, batch=1):
        """Memory-bound decode: dominated by weight loading."""
        # Weight bytes per step (approximation)
        attn_bytes = self.L * 4 * self.D * self.D * self.dtype  # Q,K,V,O projections
        ffn_bytes = self.L * 3 * self.D * 4 * self.D * self.dtype  # SwiGLU
        total_bytes = (attn_bytes + ffn_bytes)
        return total_bytes / self.hbm_bw * 1000  # per token

# DGX Spark GB10 specs
spark = InferenceTimeModel(
    'DGX Spark GB10',
    num_layers=32, d_model=4096, num_heads=32, d_head=128, num_kv_heads=8,
    peak_flops_tflops=67, hbm_bw_gb_s=273
)

print('Prefill time vs sequence length (Llama-3-8B on DGX Spark):')
for seq in [128, 256, 512, 1024, 2048, 4096]:
    t = spark.prefill_time_ms(seq)
    print(f'  seq={seq:>5}: {t:>8.2f} ms')

decode_t = spark.decode_time_ms(0)
print(f'\nDecode time per token: {decode_t:.2f} ms (memory-bound, batch=1)')
print(f'Max decode throughput: {1000/decode_t:.0f} tokens/sec (single stream)')


## 2. Optimal Worker Ratio

Given a traffic mix (mean input length $L_{in}$, mean output length $L_{out}$), what ratio of prefill:decode workers minimizes total cost while meeting SLOs?


In [ ]:
def worker_balance_analysis(mean_input_len, mean_output_len, total_workers=8):
    """
    Find optimal prefill:decode split for given traffic.
    Returns metrics for each possible split.
    """
    results = []
    for n_prefill in range(1, total_workers):
        n_decode = total_workers - n_prefill

        # Prefill capacity: requests per second each worker can handle
        t_prefill_ms = spark.prefill_time_ms(mean_input_len)
        prefill_rps = n_prefill * (1000 / t_prefill_ms)

        # Decode capacity: tokens per second each worker can handle
        t_decode_ms = spark.decode_time_ms(mean_input_len)  # approx per token
        decode_tps = n_decode * (1000 / t_decode_ms)
        decode_rps = decode_tps / mean_output_len  # requests per second

        # Bottleneck throughput
        throughput = min(prefill_rps, decode_rps)
        utilization_prefill = throughput / prefill_rps
        utilization_decode = throughput / decode_rps

        results.append({
            'n_prefill': n_prefill, 'n_decode': n_decode,
            'throughput': throughput,
            'util_prefill': utilization_prefill,
            'util_decode': utilization_decode,
        })

    return results

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (title, l_in, l_out) in zip(axes, [
    ('Short prompts (l_in=128, l_out=512)', 128, 512),
    ('Long prompts (l_in=2048, l_out=256)', 2048, 256),
]):
    results = worker_balance_analysis(l_in, l_out)
    n_p = [r['n_prefill'] for r in results]
    throughputs = [r['throughput'] for r in results]
    best_idx = np.argmax(throughputs)

    ax.plot(n_p, throughputs, 'b-o')
    ax.axvline(n_p[best_idx], color='red', linestyle='--',
               label=f'Optimal: {n_p[best_idx]} prefill, {8-n_p[best_idx]} decode')
    ax.set_xlabel('# Prefill Workers')
    ax.set_ylabel('Throughput (req/s)')
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(True)

plt.tight_layout()
plt.savefig('prefill_decode_balance.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved prefill_decode_balance.png')


## 3. TTFT vs TPOT SLO Tradeoffs

Disaggregation lets you optimize TTFT (Time to First Token) and TPOT (Time Per Output Token) independently by allocating resources to each phase.


In [ ]:
# Model TTFT and TPOT as a function of worker allocation
def slo_analysis(request_rate_rps, n_prefill, n_decode, l_in=512, l_out=256):
    """Compute expected TTFT and TPOT under M/M/c queuing approximation."""
    # Prefill system
    t_prefill = spark.prefill_time_ms(l_in) / 1000  # seconds
    prefill_load = request_rate_rps * t_prefill / n_prefill  # utilization
    if prefill_load >= 1.0:
        ttft_s = float('inf')
    else:
        ttft_s = t_prefill / (1 - prefill_load)  # M/M/1 approximation

    # Decode system
    t_decode_per_token = spark.decode_time_ms(l_in) / 1000
    total_decode_demand = request_rate_rps * l_out * t_decode_per_token
    decode_load = total_decode_demand / n_decode
    if decode_load >= 1.0:
        tpot_s = float('inf')
    else:
        tpot_s = t_decode_per_token / (1 - decode_load)

    return ttft_s * 1000, tpot_s * 1000  # ms

print('TTFT and TPOT at different worker allocations (4 total workers, 5 req/s):')
print(f'{"Prefill":>8} {"Decode":>8} {"TTFT (ms)":>12} {"TPOT (ms)":>12}')
for n_p in range(1, 4):
    n_d = 4 - n_p
    ttft, tpot = slo_analysis(5.0, n_p, n_d)
    ttft_str = f'{ttft:.0f}' if ttft < 1e6 else 'OVERlOAD'
    tpot_str = f'{tpot:.1f}' if tpot < 1e6 else 'OVERLOAD'
    print(f'{n_p:>8} {n_d:>8} {ttft_str:>12} {tpot_str:>12}')


## Experiments: Try These

1. **Traffic shift**: Simulate a traffic shift from short to long prompts. How quickly should the system reallocate workers between prefill and decode?
2. **Heterogeneous hardware**: Assign H100s to prefill (high compute) and A100s to decode (high bandwidth). Does this change the optimal ratio?
3. **Continuous profiling**: Build a monitoring tool that measures actual prefill and decode times and adjusts worker allocation dynamically.


## Key Takeaways

- Prefill-decode disaggregation enables separate optimization of TTFT (prefill-constrained) and TPOT (decode-constrained) with independent resource allocation.
- The optimal prefill:decode worker ratio depends on the input:output length ratio — short prompts/long outputs → more decode workers.
- Analytical queuing models (M/M/c) give good intuition for SLO violations before system saturation at ~70-80% utilization.
- In practice, autoscaling should continuously measure prefill and decode queue depths independently and scale each component separately.

## References
- *Inference Engineering* Ch 5.5 — Philip Kiely, Baseten Books 2026
- Patel et al. (2023), "Splitwise: Efficient Generative LLM Inference Using Phase Splitting"
- Zhong et al. (2024), "DistServe"
